# Robustness persistence — continual fine-tuning audit

Question: does adversarial robustness against problem-space attacks (VRTG, dead-code, control-flow flattening) survive a second round of fine-tuning on a non-security task, the way backdoors do (Han et al. 2026, [arXiv:2512.14741](https://arxiv.org/abs/2512.14741))?

Pipeline:
1. Score the SVD baseline on the clean corpus.
2. Generate adversarial variants with `RLAdversary`.
3. Score the model on the adversarial variants. Confidence drop = WP3 attack success.
4. (TODO, GPU) Robustness fine-tune on the adversarial variants.
5. (TODO, GPU) Continual fine-tune on a non-security task.
6. Re-score post-continual; the AUC drop is the persistence metric.

Hypothesis: if robustness is circuit-level (early attention heads), it should resist continual SFT. If it's surface-level statistical, it degrades. Han et al.'s backdoor result suggests *implanted* mappings persist; the open question is whether *emergent* robustness does too.

Steps 4-5 need GPU and aren't run here. Steps 1-3 run on CPU in under 5 minutes.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from codesign.parser import ProgramGraphExtractor
from codesign.attacker import RLAdversary
from codesign.dataset import load_samples
from codesign.metrics import summarise, AttackResult, evaded, parse_valid

SEED = 0
MAX_STEPS = 12
DATASET_ROOT = ROOT / 'data' / 'cwe_samples'
RESULTS_DIR = ROOT / 'experiments' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. SVD model

Real CodeBERT if `transformers` + `torch` are available; otherwise a heuristic so the rest of the notebook still runs.

In [ ]:
try:
    from codesign.cli import SVDTargetModel
    target = SVDTargetModel()
    target._ensure_loaded()
    score_fn = target.detect_vulnerability_score
    MODEL_TAG = target.model_id
except Exception as e:
    print(f'falling back to heuristic SVD: {e}')
    def score_fn(code: str) -> float:
        s = 0.95
        if '_adv' in code: s -= 0.40
        if '_vrtg_decoy' in code: s -= 0.30
        if 'if True:' in code: s -= 0.15
        return max(0.01, s)
    MODEL_TAG = 'heuristic-mock'
print(f'target: {MODEL_TAG}')

## 2. T0 baseline

In [ ]:
samples = load_samples(DATASET_ROOT)
print(f'loaded {len(samples)} samples; CWEs: ' + ', '.join(sorted({s.cwe for s in samples})))
baseline = {s.id: float(score_fn(s.code)) for s in samples}
for sid, score in baseline.items():
    print(f'  {sid:42s}  {score:.3f}')

## 3. T1 attack

In [ ]:
extractor = ProgramGraphExtractor()
results = []
for s in samples:
    g = extractor.build(s.code)
    adv = RLAdversary(score_fn, epsilon=0.3, seed=SEED)
    adv_code, trace = adv.attack(s.code, g.all_variable_nodes, max_steps=MAX_STEPS)
    results.append(AttackResult(
        sample_id=s.id,
        cwe=s.cwe,
        original_score=baseline[s.id],
        final_score=trace.best_score,
        steps_taken=len(trace.steps),
        evaded=evaded(trace.best_score),
        parse_valid=parse_valid(adv_code),
        dfg_preserved=False,
        mutations_applied=[step.strategy for step in trace.steps],
        final_code=adv_code,
    ))
summary = summarise(results)
print(json.dumps(summary.to_dict(), indent=2))

## 4. Robustness fine-tune (TODO, needs GPU)

Build (adversarial, label) pairs; fine-tune the SVD classifier head a few epochs with a small LR; verify the model now correctly classifies the adversarial variants. ~30 lines using `transformers.Trainer`.

In [ ]:
robust_t1 = {r.sample_id: r.final_score for r in results}

## 5. Continual fine-tune (TODO, needs GPU)

Continual SFT on docstring auto-complete (CodeSearchNet) for N steps. Expect: catastrophic forgetting of robustness if it lives in the classifier head, partial preservation if it lives in deeper attention circuits.

Han et al.'s backdoor-persistence result is the symmetric case: *implanted* mappings survive. The open question is whether emergent robustness does.

## 6. Persist

In [ ]:
import time
from dataclasses import asdict
out = RESULTS_DIR / f'robustness_forgetting_{int(time.time())}.json'
out.write_text(json.dumps({
    'schema_version': '1.0',
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'target_model': MODEL_TAG,
    'n_samples': len(samples),
    'max_steps': MAX_STEPS,
    'seed': SEED,
    'baseline_scores': baseline,
    'summary': summary.to_dict(),
    'results': [asdict(r) for r in results],
}, indent=2, default=str), encoding='utf-8')
print(f'wrote {out}')